In this notebook, we prep the catalog to be used!!

In [5]:
import os
import sys
import glob
import numpy as np
from astropy.io import fits
from astropy.table import Table, vstack, hstack
import pandas as pd
from astropy.cosmology import Planck18

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

%load_ext autoreload
%autoreload 2



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# load the MAIN extension directly as an Astropy Table
data_cat = Table.read(filename, hdu="MAIN")


In [7]:
fspec_cat = Table.read(filename, hdu="FASTSPEC")


In [8]:
spec_cat = Table.read(filename, hdu="SPECTRA_TEMPLATE")


In [9]:

mask = (data_cat["DWARF_MASKBIT"] == 0)

data_cat = data_cat[mask]
fspec_cat = fspec_cat[mask]
spec_cat = spec_cat[mask]


In [10]:
len(data_cat)

439514

In [11]:
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import pandas as pd

In [12]:
# If you want TARGETID stored as a string instead of int64+BigInt, flip this.
TARGETID_AS_STRING = False


def _clean_sample(s):
    """Turn b'BGS_BRIGHT' / bytes / np.bytes_ into a plain 'BGS_BRIGHT' string."""
    if isinstance(s, (bytes, np.bytes_)):
        return s.decode("utf-8", "ignore")
    if isinstance(s, str) and s.startswith("b'") and s.endswith("'"):
        return s[2:-1]
    return s


def _prep_targetid(df):
    if TARGETID_AS_STRING:
        df["TARGETID"] = df["TARGETID"].astype("int64").astype(str)
    else:
        df["TARGETID"] = df["TARGETID"].astype("int64")
    return df


def write_parquet(df, path, *, row_group_size=100_000, compression="snappy"):
    """Write a DataFrame to a browser-friendly Parquet file.

    - snappy compression: natively supported by hyparquet, no extra JS deps.
      (Use 'zstd' for ~20-30% smaller files, but then the frontend must register
       hyparquet-compressors — see notes at the bottom.)
    - row groups of 100k keep the file friendly for future range-request queries
      (DuckDB-WASM / Tier 2) without hurting full-file reads now.
    """
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(
        table,
        path,
        compression=compression,
        row_group_size=row_group_size,
        use_dictionary=True,   # great for SAMPLE and the 0/1 flag columns
        version="2.6",
    )
    print(f"wrote {path}  ({pq.read_metadata(path).num_rows:,} rows)")

In [37]:
main_cat = data_cat["TARGETID","Z","RA","DEC","LOG_MSTAR_M24", "MAG_R","SAMPLE", "R50_R"]

ba_val = data_cat["SHAPE_PARAMS"][:,0]

pa_val = data_cat["SHAPE_PARAMS"][:,1]

main_cat["BA"] = ba_val
main_cat["PA"] = pa_val


In [40]:

#add a new column for distance
# main_cat["DIST_MPC"] = Planck18.comoving_distance(main_cat["Z"]).value

# spec_cat = spec_cat["SPEC_UMAP_0", "SPEC_UMAP_1"]

tot_cat = hstack([main_cat, fspec_cat["HALPHA_FLUX"]])

# tot_cat = tot_cat[tot_cat["SPEC_UMAP_0"] > -50 ]


In [47]:
tot_cat[:2].columns

<TableColumns names=('TARGETID','Z','RA','DEC','LOG_MSTAR_M24','MAG_R','SAMPLE','R50_R','BA','PA','HALPHA_FLUX')>

In [46]:
# tot_cat is an astropy Table -> convert to a pandas DataFrame
main = tot_cat.to_pandas()

main["SAMPLE"] = main["SAMPLE"].map(_clean_sample).astype("category")
main = _prep_targetid(main)
main = main.astype({
    "Z":             "float32",
    "RA":            "float64",
    "DEC":           "float64",
    "LOG_MSTAR_M24": "float32",
    "MAG_R":         "float32",
    "HALPHA_FLUX":   "float32",
    "BA":            "float32",
    "PA":            "float32",
    "R50_R":         "float32",
})
write_parquet(main, "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dwarfs.parquet")


wrote /pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dwarfs.parquet  (439,514 rows)


In [18]:
# # Convert all columns except TARGETID and SAMPLE to float16
# for col in tot_cat.colnames:
#     if col not in ["TARGETID", "SAMPLE"]:
#         if col in ["RA","DEC", "DIST_MPC"]:
#             tot_cat[col] = tot_cat[col].astype('float32')
#         else:
#             tot_cat[col] = tot_cat[col].astype('float16')

# # Compute total size in bytes
# total_bytes = sum(tot_cat[col].nbytes for col in tot_cat.colnames)
# total_mb = total_bytes / (1024**2)
# print(f"Total catalog size: {total_mb:.2f} MB")

# # Convert to Pandas DataFrame and save as CSV
# df = tot_cat.to_pandas()

# df.to_csv("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/desi_dwarfs.csv", index=False)


In [84]:
spec_cat['SPEC_UMAP_0'] = spec_cat['SPEC_UMAP_0'].astype(np.float32)
spec_cat['SPEC_UMAP_1'] = spec_cat['SPEC_UMAP_1'].astype(np.float32)

#  Convert NNMF_RESID to float16
spec_cat['NNMF_RESID'] = spec_cat['NNMF_RESID'].astype(np.float16)

# Convert SAMPLE string column to 4 binary columns
samples = ['ELG', 'BGS_BRIGHT', 'BGS_FAINT', 'LOWZ']

# Initialize new columns with zeros
for s in samples:
    col_name = f'in_{s.replace("BGS_BRIGHT","BGSB").replace("BGS_FAINT","BGSF")}'
    spec_cat[col_name] = np.zeros(len(spec_cat), dtype=np.int8)  # 0/1 column

# Fill in ones according to SAMPLE
for i, row in enumerate(spec_cat):
    if row['SAMPLE'] == 'ELG':
        spec_cat['in_ELG'][i] = 1
    elif row['SAMPLE'] == 'BGS_BRIGHT':
        spec_cat['in_BGSB'][i] = 1
    elif row['SAMPLE'] == 'BGS_FAINT':
        spec_cat['in_BGSF'][i] = 1
    elif row['SAMPLE'] == 'LOWZ':
        spec_cat['in_LOWZ'][i] = 1

# Remove the original SAMPLE column if you like
spec_cat.remove_column('SAMPLE')



In [85]:
df_spec = spec_cat.to_pandas()
df_spec.to_csv("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/umap_catalog.csv", index=False)


In [86]:
df_spec

,TARGETID,SPEC_UMAP_0,SPEC_UMAP_1,NNMF_RESID,in_ELG,in_BGSB,in_BGSF,in_LOWZ
0,39627066986994125,12.842730,4.534079,51.56250,0,1,0,0
1,39627067007963313,12.498918,2.934067,53.62500,0,1,0,0
2,39627072187928583,6.793174,4.161978,54.90625,0,1,0,0
3,39627072192121852,10.536270,-0.548725,52.81250,0,1,0,0
4,39627077380476509,10.724737,0.587644,52.96875,0,1,0,0
...,...,...,...,...,...,...,...,...
350623,39627636598641999,11.761548,1.646974,57.65625,0,1,0,0
350624,39627769058951413,8.642703,9.014146,80.18750,0,1,0,0
350625,39627351838953626,9.755444,2.125329,57.84375,0,1,0,0
350626,39627764344553699,10.637001,1.671248,66.00000,0,1,0,0


In [48]:
## get the catalog of massive galaxies for reference!!



In [67]:
bgsb_cat = Table.read("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/iron_bgs_bright_filter_zsucc_zrr02_allfracflux_INT.fits")

In [68]:
bgsb_cat_f = bgsb_cat[ (bgsb_cat["FRACFLUX_R"] < 0.35) & (bgsb_cat["Z"] < 0.2) & (bgsb_cat["DELTACHI2"] > 40) ]

In [69]:
### get the approximate absolute magnitude sample!

In [70]:
from astropy.cosmology import Planck18

In [71]:
bgsb_cat_absmag_r = bgsb_cat_f["MAG_R"].data + 5 - 5*np.log10(Planck18.luminosity_distance(bgsb_cat_f["Z"].data).value * 1e6)

In [ ]:
bgsb_cat_massive_mw = bgsb_cat_f[ (bgsb_cat_absmag_r < -21.5) & (bgsb_cat_f["Z"] < 0.1) & (bgsb_cat_f["Z"] > 0.001) ]["RA","DEC","Z"]

##here I just need RA,DEC,Z